# TELCO CUSTOMER CHURN PREDICTION — 5-Model Comparative Study

**Dataset:** IBM Telco Customer Churn (standard Kaggle version)

**Models Evaluated:**
1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM (Gradient Boosting)
5. Support Vector Machine (SVM)


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

In [ ]:
# ── INSTALL DEPENDENCIES ────────────────────────────────────────────────────────
!pip install xgboost lightgbm xlrd --quiet

In [ ]:
# ── 1. IMPORTS ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("✅ All imports successful.")

In [ ]:
# ── 2. LOAD DATA ────────────────────────────────────────────────────────────────
df = pd.read_excel("/content/WA_Fn-UseC_-Telco-Customer-Churn.xls")

print(f"✅ Data loaded  →  {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# ── 3. DATA CLEANING ─────────────────────────────────────────────────────────────

# 3a. Drop customerID — unique identifier with zero predictive signal
df.drop(columns=["customerID"], inplace=True)

# 3b. Safely coerce TotalCharges to numeric (whitespace strings → NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 3c. Impute TotalCharges NaN with median (only 11 rows; all tenure=0)
median_tc = df["TotalCharges"].median()
df["TotalCharges"].fillna(median_tc, inplace=True)
print(f"   TotalCharges NaN filled with median = ${median_tc:,.2f}")

# 3d. Encode binary target: Yes → 1, No → 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# 3e. One-hot encode all remaining object columns
df = pd.get_dummies(df, drop_first=True)
print(f"   After encoding  →  {df.shape[1]} feature columns")

In [ ]:
# ── 4. FEATURE / TARGET SPLIT ────────────────────────────────────────────────────
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [ ]:
# ── 5. TRAIN / TEST SPLIT (80 / 20, stratified) ────────────────────────────────
#     Stratify ensures both sets have the same churn ratio (~26%).
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print(f"\n✅ Train size: {X_train.shape[0]:,}  |  Test size: {X_test.shape[0]:,}")
print(f"   Churn rate  →  train: {y_train.mean():.2%}  |  test: {y_test.mean():.2%}")

In [ ]:
# ── 6. FEATURE SCALING ──────────────────────────────────────────────────────────
#     Scaler fitted ONLY on X_train to prevent data leakage.
#     Required for Logistic Regression and SVM; harmless for tree-based models.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Features scaled (StandardScaler).")

In [ ]:
# ── 7. DEFINE & TRAIN ALL 5 MODELS ─────────────────────────────────────────────
#
#   class_weight='balanced' / scale_pos_weight corrects for class imbalance.
#   Churn rate is ~26%, so the non-churn class is ~3x more frequent.

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count   # ≈ 2.77

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}

results = {}

for name, model in models.items():
    print(f"\n🔄 Training: {name}")
    model.fit(X_train_scaled, y_train)
    y_pred      = model.predict(X_test_scaled)
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        "model":      model,
        "y_pred":     y_pred,
        "y_pred_prob":y_pred_prob,
        "Accuracy":   accuracy_score(y_test, y_pred),
        "Precision":  precision_score(y_test, y_pred),
        "Recall":     recall_score(y_test, y_pred),
        "F1-Score":   f1_score(y_test, y_pred),
        "ROC-AUC":    roc_auc_score(y_test, y_pred_prob),
    }
    print(f"   ✅ Done — Recall: {results[name]['Recall']:.4f}  |  ROC-AUC: {results[name]['ROC-AUC']:.4f}")

print("\n🏁 All 5 models trained successfully!")

In [ ]:
# ── 8. RESULTS SUMMARY TABLE (sorted by Recall) ─────────────────────────────────

metrics_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

rows = []
for name, r in results.items():
    rows.append({
        "Model":     name,
        "Accuracy":  round(r["Accuracy"],  4),
        "Precision": round(r["Precision"], 4),
        "Recall":    round(r["Recall"],    4),
        "F1-Score":  round(r["F1-Score"],  4),
        "ROC-AUC":   round(r["ROC-AUC"],   4),
    })

summary_df = (
    pd.DataFrame(rows)
      .set_index("Model")
      .sort_values("Recall", ascending=False)
)

print("\n" + "="*70)
print("  5-MODEL PERFORMANCE SUMMARY — TELCO CHURN PREDICTION")
print("  (Sorted by Recall, Churn class)")
print("="*70)
print(summary_df.to_string())
print("="*70)

In [ ]:
# ── 9. VISUALISATIONS ───────────────────────────────────────────────────────────
%matplotlib inline
from IPython.display import display

# ── Shared Style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

PALETTE     = ["#1A73E8", "#34A853", "#EA4335", "#FBBC04", "#9C27B0"]
BRAND_GREY  = "#5F6368"
BRAND_LIGHT = "#F8F9FA"

model_names = list(results.keys())

In [ ]:
# ── PLOT 1: Combined ROC Curve (all 5 models on one plot) ─────────────────────
print("📊 PLOT 1 OF 3 — Combined ROC Curve")

fig1, ax1 = plt.subplots(figsize=(8, 6), facecolor=BRAND_LIGHT)

for idx, (name, r) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, r["y_pred_prob"])
    ax1.plot(
        fpr, tpr,
        color=PALETTE[idx], lw=2.2,
        label=f"{name} (AUC = {r['ROC-AUC']:.3f})"
    )
    ax1.fill_between(fpr, tpr, alpha=0.04, color=PALETTE[idx])

ax1.plot([0, 1], [0, 1], linestyle="--", color=BRAND_GREY, lw=1.2, label="Random")
ax1.set_xlabel("False Positive Rate", fontsize=11)
ax1.set_ylabel("True Positive Rate", fontsize=11)
ax1.set_title("Combined ROC Curve — All 5 Models", fontweight="bold", pad=14, fontsize=13)
ax1.legend(loc="lower right", fontsize=9.5)
ax1.grid(linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("plot1_combined_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → plot1_combined_roc_curve.png\n")

In [ ]:
# ── PLOT 2: Seaborn Bar Chart — Recall Comparison ─────────────────────────────
print("📊 PLOT 2 OF 3 — Recall Score Comparison")

recall_vals  = [results[m]["Recall"] for m in model_names]
sorted_pairs = sorted(zip(recall_vals, model_names), reverse=True)
sorted_recalls, sorted_names = zip(*sorted_pairs)

fig2, ax2 = plt.subplots(figsize=(9, 5.5), facecolor=BRAND_LIGHT)

bars = sns.barplot(
    x=list(sorted_names),
    y=list(sorted_recalls),
    palette=PALETTE,
    ax=ax2
)

ax2.set_ylim(0, 1.15)
ax2.set_title("Recall Score by Model (Churn Class)", fontweight="bold", pad=14, fontsize=13)
ax2.set_xlabel("Model", fontsize=11)
ax2.set_ylabel("Recall Score", fontsize=11)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax2.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax2.set_axisbelow(True)

for bar, val in zip(ax2.patches, sorted_recalls):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.2%}",
        ha="center", va="bottom",
        fontsize=10, fontweight="bold", color=BRAND_GREY
    )

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("plot2_recall_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → plot2_recall_comparison.png\n")

In [ ]:
# ── PLOT 3: Full Metrics Heatmap Table ─────────────────────────────────────────
print("📊 PLOT 3 OF 3 — Full Metrics Heatmap")

heat_df = summary_df.copy()   # already sorted by Recall

fig3, ax3 = plt.subplots(figsize=(10, 4.5), facecolor=BRAND_LIGHT)

sns.heatmap(
    heat_df,
    annot=True,
    fmt=".4f",
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="white",
    ax=ax3,
    vmin=0.45, vmax=0.95,
    annot_kws={"size": 10}
)

ax3.set_title(
    "Model Performance Metrics — Sorted by Recall",
    fontweight="bold", pad=14, fontsize=13
)
ax3.set_xlabel("Metric", fontsize=11)
ax3.set_ylabel("Model", fontsize=11)
ax3.tick_params(axis="x", rotation=0)
ax3.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("plot3_metrics_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → plot3_metrics_heatmap.png\n")

In [ ]:
# ── 10. FINAL SUMMARY ───────────────────────────────────────────────────────────

best_recall_model = summary_df["Recall"].idxmax()
best_auc_model    = summary_df["ROC-AUC"].idxmax()
best_f1_model     = summary_df["F1-Score"].idxmax()

print("\n" + "="*60)
print("  EXECUTIVE SUMMARY")
print("="*60)
print(f"  🥇 Best Recall  : {best_recall_model}")
print(f"      Recall  = {summary_df.loc[best_recall_model, 'Recall']:.4f}")
print(f"  🏆 Best ROC-AUC : {best_auc_model}")
print(f"      ROC-AUC = {summary_df.loc[best_auc_model, 'ROC-AUC']:.4f}")
print(f"  🎯 Best F1-Score: {best_f1_model}")
print(f"      F1-Score = {summary_df.loc[best_f1_model, 'F1-Score']:.4f}")
print("="*60)
print("\n  Full Leaderboard (sorted by Recall):")
print(summary_df.to_string())
print("="*60)